# 02 — Fusion & traitement StockTwits 2020-2022

**Objectif** : exploiter le corpus intraday `StockTwits_2020_2022_Raw` (~4.2 M messages, résolution minute) pour construire un signal textuel **causal** de prédiction boursière.

**Période** : 2020-01-01 → 2022-03-05 (fin de la collecte)
**Tickers exploitables** (parmi notre panel) : `AMZN`, `TSLA`
**Prix de référence** : `dataset_finance_tiingo_2010_2026.csv`

> **Causalité** : la fenêtre « overnight » (après clôture J−1 + avant 9h30 J) est **connue avant l'ouverture de J**. On prédit le rendement du jour J (design A) — seule utilisation légale du signal, contrairement au lag-0 contemporain.

## Plan
1. Chargement et parsing des 362 CSV (~4.2 M messages)
2. Nettoyage, horodatage NY, approbation par ticker
3. Sentiment VADER par message
4. Agrégation en panel quotidien (fenêtres morn / market / post)
5. Fusion prix tiingo + features overnight (pré-ouverture)
6. EDA & corrélations
7. Walk-forward strict (LightGBM vs naïf)
8. Validation FinBERT sur un échantillon


## 1. Configuration

In [2]:
import numpy as np
import pandas as pd
import glob, os, re
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from datetime import datetime
import gc 
sns.set_theme(style="whitegrid")
np.random.seed(42)

BASE = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\raw\source_data_trouve_stocktwits\StockTwits_2020_2022_Raw"
OUT_DIR = r"C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed"
print("Base StockTwits :", BASE)
print("Repertoire sortie :", OUT_DIR)
print("Existe :", os.path.isdir(BASE))
print("Date d'execution :", datetime.now().strftime('%Y-%m-%d %H:%M'))

Base StockTwits : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\raw\source_data_trouve_stocktwits\StockTwits_2020_2022_Raw
Repertoire sortie : C:\Users\semy4\OneDrive\Bureau\Fintech_project\data\processed
Existe : True
Date d'execution : 2026-08-31 22:38


### Vérification packages

In [42]:
import importlib
for m in ['vaderSentiment', 'transformers', 'torch', 'lightgbm', 'sklearn']:
    try:
        importlib.import_module(m)
        print(f"  {m:20s} OK")
    except Exception as e:
        print(f"  {m:20s} MANQUANT -> {type(e).__name__}")

  vaderSentiment       OK
  transformers         OK
  torch                OK
  lightgbm             OK
  sklearn              OK


## Fusion des datasets

In [43]:
FOLDERS = ["AAPL_2020_2022", "AMZN2019-2022", "FB_2019_2022", "NVDA_2013_2022", "TSLA_2020_2022"]

In [44]:
csv_files = []
for d in FOLDERS:
    csv_files += glob.glob(os.path.join(BASE, d, '*.csv'))
print(f"Nombre de fichiers à traiter : {len(csv_files)}")

Nombre de fichiers à traiter : 362


In [45]:
# 2. Préparation des Regex pour extraire le ticker (identique à ton fichier source)
SYM_RE = re.compile(r"['\"]symbol['\"]\s*[:=]\s*['\"]([A-Z]+)['\"]")
CASHTAG_RE = re.compile(r'\$([A-Z]{1,5})\b')
PANEL_T = {'AAPL', 'AMZN', 'META', 'NVDA', 'TSLA'} # Les tickers qu'on veut garder

In [46]:
def extraire_donnees(df):
    """Extrait le ticker et ne garde que Date, Tweet et Ticker"""
    # Recherche du ticker dans 'symbols', sinon dans le texte du tweet ('body')
    sym = df['symbols'].fillna('').astype(str).str.extract(SYM_RE, expand=False).fillna('')
    cashtag = df['body'].fillna('').astype(str).str.extract(CASHTAG_RE, expand=False).fillna('')
    
    tick = sym.where(sym != '', cashtag)
    tick = tick.map(lambda s: 'META' if s == 'FB' else s) # FB est devenu META
    tick = tick.where(tick.isin(PANEL_T))
    
    df['Ticker'] = tick
    
    # On ne garde que les 3 colonnes demandées
    df_propre = df[['created_at', 'body', 'Ticker']]
    
    # On supprime les lignes vides (sans date ou sans ticker reconnu)
    return df_propre.dropna(subset=['created_at', 'Ticker'])

In [47]:
# 3. Boucle de traitement fichier par fichier
chunks = []
for f in csv_files:
    try:
        # Lire juste les en-têtes pour repérer les colonnes utiles
        cols = pd.read_csv(f, nrows=0).columns
        keep = [c for c in cols if c.lower() in {'created_at', 'body', 'symbols'}]
        
        # Charger uniquement les colonnes nécessaires (très important pour la RAM)
        df = pd.read_csv(f, usecols=keep)
        
        # Appliquer notre fonction de nettoyage
        df_nettoye = extraire_donnees(df)
        chunks.append(df_nettoye)
        
        # Libérer la mémoire
        del df
    except Exception as e:
        print(f"Erreur sur le fichier {f} : {e}")

# 4. Assembler le tout dans un seul grand tableau
mon_dataset = pd.concat(chunks, ignore_index=True)

# Vider la mémoire des morceaux
del chunks
gc.collect()

# 5. Renommer pour que ce soit propre et formater la date
mon_dataset = mon_dataset.rename(columns={'created_at': 'Date', 'body': 'Tweet'})
mon_dataset['Date'] = pd.to_datetime(mon_dataset['Date'], utc=True, errors='coerce')

print(f"\nTerminé ! Taille finale du dataset : {len(mon_dataset)} lignes.")
display(mon_dataset.head())


Terminé ! Taille finale du dataset : 4322378 lignes.


,Date,Tweet,Ticker
0,2020-01-03 20:14:21+00:00,$AAPL there is some smarty money trimming into...,AAPL
1,2020-01-03 20:13:15+00:00,$AAPL boom!,AAPL
2,2020-01-03 20:12:42+00:00,$AAPL this pos will not fall below mid VWAP.,AAPL
3,2020-01-03 20:12:31+00:00,$AAPL I am holding going to see what this bab...,AAPL
4,2020-01-03 20:10:08+00:00,$SPY is $AAPL over priced is the average analy...,AAPL


In [48]:
mon_dataset.tail()

,Date,Tweet,Ticker
4322373,2019-12-24 20:53:33+00:00,$TSLA This is why TESLA will always STAND OUT ...,TSLA
4322374,2019-12-24 20:49:34+00:00,$TSLA oh boy! Going to ROCKET Thursday! 450 by...,TSLA
4322375,2019-12-24 20:46:39+00:00,$SHAK sorry Morgan Stanley your price targets ...,TSLA
4322376,2019-12-24 20:45:49+00:00,$TSLA it so painful holding puts over Xmas but...,TSLA
4322377,2019-12-24 20:45:19+00:00,$TSLA 🤙,TSLA


In [49]:
mon_dataset["Ticker"].nunique()

5

In [50]:

print("Valeurs manquantes avant nettoyage :")
print(mon_dataset.isna().sum())


mon_dataset = mon_dataset.dropna(subset=['Date', 'Tweet'])

print(f"\nTaille après suppression des manquants : {len(mon_dataset)}")

Valeurs manquantes avant nettoyage :
Date      0
Tweet     0
Ticker    0
dtype: int64

Taille après suppression des manquants : 4322378


In [51]:
# 1. Compter les lignes strictement identiques (même date, même tweet, même ticker)
nb_doublons = mon_dataset.duplicated().sum()
print(f"Nombre de lignes totalement dupliquées : {nb_doublons}")

# 2. Supprimer ces doublons en conservant uniquement la première apparition
if nb_doublons > 0:
    mon_dataset = mon_dataset.drop_duplicates(keep='first')
    # On réinitialise l'index pour avoir une numérotation propre et continue
    mon_dataset = mon_dataset.reset_index(drop=True)

print(f"Taille finale du dataset propre : {len(mon_dataset)}")

Nombre de lignes totalement dupliquées : 206283
Taille finale du dataset propre : 4116095


In [52]:
# 1. Convertir le fuseau horaire EN PREMIER (UTC -> New York)
mon_dataset['Date_NY'] = mon_dataset['Date'].dt.tz_convert('America/New_York')

# 2. Filtrer les années sur l'heure locale de New York
mon_dataset = mon_dataset[mon_dataset['Date_NY'].dt.year.isin([2020, 2021, 2022])]
mon_dataset = mon_dataset.reset_index(drop=True)

# 3. Extraire le jour et l'heure décimale
mon_dataset['Jour'] = mon_dataset['Date_NY'].dt.date.astype(str)
mon_dataset['Heure_decimale'] = mon_dataset['Date_NY'].dt.hour + mon_dataset['Date_NY'].dt.minute / 60.0

# 4. Nettoyer les colonnes
mon_dataset = mon_dataset.drop(columns=['Date', 'Date_NY'])

# 5. Forcer le tri chronologique absolu (Ticker, puis Jour, puis Heure)
mon_dataset = mon_dataset.sort_values(by=['Ticker', 'Jour', 'Heure_decimale']).reset_index(drop=True)

print(f"Lignes restantes (2020-2022, heure NY stricte) : {len(mon_dataset)}")
print(f"Plage temporelle : {mon_dataset['Jour'].min()} -> {mon_dataset['Jour'].max()}")
display(mon_dataset.head())

Lignes restantes (2020-2022, heure NY stricte) : 3711333
Plage temporelle : 2020-01-01 -> 2022-03-05


,Tweet,Ticker,Jour,Heure_decimale
0,$AAPL good things happening 2020 run trump and...,AAPL,2020-01-01,0.100000
1,$AAPL Happy New Year amazing winning AAPL Bull...,AAPL,2020-01-01,0.133333
2,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,AAPL,2020-01-01,0.233333
3,@Taxes_R2_Damn_High my dad ended this year by...,AAPL,2020-01-01,0.316667
4,$AAPL And to those using the tired and old del...,AAPL,2020-01-01,0.616667


In [53]:
# 1. Tri chronologique strict
mon_dataset = mon_dataset.sort_values(by=['Ticker', 'Jour', 'Heure_decimale'])

# 2. Réinitialisation de l'index pour avoir une numérotation propre de 0 à la fin
mon_dataset = mon_dataset.reset_index(drop=True)

# 3. Vérification visuelle
print("Aperçu des premières lignes après le tri chronologique :")
display(mon_dataset.head(10))

Aperçu des premières lignes après le tri chronologique :


,Tweet,Ticker,Jour,Heure_decimale
0,$AAPL good things happening 2020 run trump and...,AAPL,2020-01-01,0.100000
1,$AAPL Happy New Year amazing winning AAPL Bull...,AAPL,2020-01-01,0.133333
2,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,AAPL,2020-01-01,0.233333
3,@Taxes_R2_Damn_High my dad ended this year by...,AAPL,2020-01-01,0.316667
4,$AAPL And to those using the tired and old del...,AAPL,2020-01-01,0.616667
5,$AAPL **&quot;With massively more room to grow...,AAPL,2020-01-01,0.683333
6,$AAPL gonna rip on Thursday for the new year,AAPL,2020-01-01,1.000000
7,$AAPL $MSFT $AMZN $PTI bulls and bears.. Happy...,AAPL,2020-01-01,1.066667
8,"Bullish investment portfolio: Apple($AAPL), In...",AAPL,2020-01-01,1.200000
9,"$AAPL can’t get AirPod pros anywhere, crazy stuff",AAPL,2020-01-01,2.066667


In [54]:
mon_dataset.to_csv(os.path.join(OUT_DIR, 'StockTwits_2020_2022_cleaned.csv'), index=False)

## test de VADER

In [55]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


analyzer = SentimentIntensityAnalyzer()

def calculer_vader(texte):
    try:
        # Le score 'compound' normalise le sentiment entre -1 (très négatif) et +1 (très positif)
        return analyzer.polarity_scores(str(texte))['compound']
    except Exception:
        return 0.0

# 1. Isoler un petit échantillon (15 lignes par exemple, tu peux changer ce nombre)
echantillon = mon_dataset.sample(n=15, random_state=42).copy()

# 2. Appliquer la fonction de scoring uniquement sur l'échantillon
echantillon['Score_VADER'] = echantillon['Tweet'].map(calculer_vader)

# 3. Afficher le Ticker, le Tweet et son score pour vérifier la pertinence
display(echantillon[['Ticker', 'Tweet', 'Score_VADER']])

,Ticker,Tweet,Score_VADER
297508,AAPL,$AAPL $MSFT come on like this post fam,0.3612
895984,AAPL,$AAPL stock analysis based on today&#39;s clos...,0.0000
2478845,TSLA,@Jholek you have no idea. Hope this company sc...,-0.6814
85710,AAPL,$AAPL I can&#39;t decide wether to be bearish ...,0.0000
864748,AAPL,$AAPL Wells Fargo maintains. PT $205,0.2500
3010298,TSLA,$TSLA Gordon Johnson is super lame,0.2732
1927322,TSLA,$TSLA star link should be interesting componen...,0.4019
2343679,TSLA,$TSLA PUSH!!! $2175 EOD...PUSH MFER!!!,0.0000
361511,AAPL,$AAPL,0.0000
3516915,TSLA,$TSLA more than scary,-0.5390


## 🎯 Conclusion : Limites de VADER sur le jargon financier

L'analyse de cet échantillon aléatoire met en évidence les limites d'une approche lexicale généraliste comme VADER pour le traitement du langage naturel (NLP) appliqué à la finance. Plusieurs erreurs de classification structurelles ressortent :

* **Faux neutres sur les signaux transactionnels :** Des directives claires comme `$TSLA sell.sell.sell.` sont évaluées avec un score de `0.0000`. Dans le langage courant, "vendre" n'est pas un mot à polarité négative, mais sur les marchés, c'est un signal fortement baissier (*bearish*).
* **Ignorance de la terminologie technique :** L'analyseur ne reconnaît pas le vocabulaire des produits dérivés. Un tweet annonçant l'achat de `CALLS` (options d'achat pariant sur la hausse) est classé comme neutre, omettant un signal haussier (*bullish*) évident.
* **Incapacité à capter le contexte et l'ironie :** Les scénarios hypothétiques ou le sarcasme ("imagine if... bombed") génèrent des faux positifs, le modèle se basant uniquement sur la somme des mots individuels sans comprendre la structure de la phrase.

**Conclusion intermédiaire :** 
L'utilisation de VADER entraîne une perte d'information majeure sur notre corpus. Pour construire un signal de trading robuste, il est indispensable de s'orienter vers des modèles de *Deep Learning* contextuels, pré-entraînés spécifiquement sur des corpus financiers (comme **FinBERT**).

In [3]:
mon_dataset=pd.read_csv(os.path.join(OUT_DIR, 'StockTwits_2020_2022_cleaned.csv'))

In [4]:
import re
import html

def nettoyer_texte_finbert(texte):
    # Sécurité : vérifier que c'est bien une chaîne de caractères
    if not isinstance(texte, str):
        return ""
    
    # 1. Convertir les entités HTML (ex: &amp; -> &)
    texte = html.unescape(texte)
    
    # 2. Supprimer les liens (http, https, www)
    texte = re.sub(r'http\S+|www\S+|https\S+', '', texte, flags=re.MULTILINE)
    
    # 3. Supprimer les mentions (@utilisateur)
    texte = re.sub(r'\@\w+', '', texte)
    
    # 4. Nettoyer les espaces multiples et sauts de ligne
    texte = re.sub(r'\s+', ' ', texte).strip()
    
    return texte

print("⏳ Lancement du nettoyage des tweets (cela peut prendre un peu de temps vu le volume)...")

# Application de la fonction sur la colonne 'Tweet' de notre dataset 2020-2022
mon_dataset['Texte_Nettoye'] = mon_dataset['Tweet'].apply(nettoyer_texte_finbert)

print("✅ Nettoyage terminé !")

# Affichage pour vérifier l'avant/après
display(mon_dataset[['Ticker', 'Tweet', 'Texte_Nettoye']].head(10))

⏳ Lancement du nettoyage des tweets (cela peut prendre un peu de temps vu le volume)...
✅ Nettoyage terminé !


,Ticker,Tweet,Texte_Nettoye
0,AAPL,$AAPL good things happening 2020 run trump and...,$AAPL good things happening 2020 run trump and...
1,AAPL,$AAPL Happy New Year amazing winning AAPL Bull...,$AAPL Happy New Year amazing winning AAPL Bull...
2,AAPL,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,Happy New Year :) $AAPL $TSLA $AMZN $SPY $BTC.X
3,AAPL,@Taxes_R2_Damn_High my dad ended this year by...,my dad ended this year by selling half his $aa...
4,AAPL,$AAPL And to those using the tired and old del...,$AAPL And to those using the tired and old del...
5,AAPL,$AAPL **&quot;With massively more room to grow...,"$AAPL **""With massively more room to grow"" not..."
6,AAPL,$AAPL gonna rip on Thursday for the new year,$AAPL gonna rip on Thursday for the new year
7,AAPL,$AAPL $MSFT $AMZN $PTI bulls and bears.. Happy...,$AAPL $MSFT $AMZN $PTI bulls and bears.. Happy...
8,AAPL,"Bullish investment portfolio: Apple($AAPL), In...","Bullish investment portfolio: Apple($AAPL), In..."
9,AAPL,"$AAPL can’t get AirPod pros anywhere, crazy stuff","$AAPL can’t get AirPod pros anywhere, crazy stuff"


In [5]:
mon_dataset

,Tweet,Ticker,Jour,Heure_decimale,Texte_Nettoye
0,$AAPL good things happening 2020 run trump and...,AAPL,2020-01-01,0.100000,$AAPL good things happening 2020 run trump and...
1,$AAPL Happy New Year amazing winning AAPL Bull...,AAPL,2020-01-01,0.133333,$AAPL Happy New Year amazing winning AAPL Bull...
2,Happy New Year :)\n$AAPL $TSLA $AMZN $SPY $B...,AAPL,2020-01-01,0.233333,Happy New Year :) $AAPL $TSLA $AMZN $SPY $BTC.X
3,@Taxes_R2_Damn_High my dad ended this year by...,AAPL,2020-01-01,0.316667,my dad ended this year by selling half his $aa...
4,$AAPL And to those using the tired and old del...,AAPL,2020-01-01,0.616667,$AAPL And to those using the tired and old del...
...,...,...,...,...,...
3711328,$TSLA 777,TSLA,2022-02-28,20.783333,$TSLA 777
3711329,$AMC $TSLA \n\nCan’t wait to buy a Tesla for o...,TSLA,2022-02-28,20.800000,$AMC $TSLA Can’t wait to buy a Tesla for overa...
3711330,$TSLA with future up now. any guess what will ...,TSLA,2022-02-28,20.800000,$TSLA with future up now. any guess what will ...
3711331,$TSLA what are the chances of Biden mentioning...,TSLA,2022-02-28,20.916667,$TSLA what are the chances of Biden mentioning...


In [6]:
mon_dataset.to_csv(os.path.join(OUT_DIR, 'StockTwits_2020_2022_cleaned_final.csv'), index=False)